<a href="https://colab.research.google.com/github/maryamashraf6565-star/bank_loaner_decider/blob/main/bank_loaning_decider.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

# 1. Create the data dictionary (Notice "Me" is left blank/NaN)
data = {
    'Customer': ['Customer 1', 'Customer 2', 'Customer 3', 'Customer 4', 'Customer 5', 'Me'],
    'Age (years)': [16, 24, 23, 28, 40, 50],
    'Salary (monthly)': [0, 5000, 6000, 5000, 15000, 12000],
    'Debt (unit: 10,000)': [0, 0, 10, 50, 200, 30],
    'Approved': ['No', 'Yes', 'Yes', 'No', 'Yes', None]
}

# 2. Convert to a pandas DataFrame
df = pd.DataFrame(data)

# 3. Display the table to make sure it looks right
print("--- Dataset Preview ---")
display(df)

--- Dataset Preview ---


,Customer,Age (years),Salary (monthly),"Debt (unit: 10,000)",Approved
0,Customer 1,16,0,0,No
1,Customer 2,24,5000,0,Yes
2,Customer 3,23,6000,10,Yes
3,Customer 4,28,5000,50,No
4,Customer 5,40,15000,200,Yes
5,Me,50,12000,30,None


In [4]:
import numpy as np
# 1. Separate Features (X) and Target (y)
# Drop the 'Customer' name column and the 'Approved' column to get just the 3 features
x=df.drop(columns=["Customer","Approved"])
y=df.drop(columns=["Customer","Age (years)","Salary (monthly)","Debt (unit: 10,000)"])
# 2. Split into Training Data and the "Me" data we want to predict
# The last row is "Me", so we separate it out
X_train = x.iloc[:-1] # All rows except the last one
y_train = y.iloc[:-1]
X_me = x.iloc[-1:]    # Only the last row

print("Training Data (X):")
display(X_train)
print("Target Labels (y):")
display(y_train)
print("My Data to Predict:")
display(X_me)

Training Data (X):


,Age (years),Salary (monthly),"Debt (unit: 10,000)"
0,16,0,0
1,24,5000,0
2,23,6000,10
3,28,5000,50
4,40,15000,200


Target Labels (y):


,Approved
0,No
1,Yes
2,Yes
3,No
4,Yes


My Data to Predict:


,Age (years),Salary (monthly),"Debt (unit: 10,000)"
5,50,12000,30


In [5]:

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# --- 1. Load Data ---
data = {
    'Age (years)': [16, 24, 23, 28, 40, 50],
    'Salary (monthly)': [0, 5000, 6000, 5000, 15000, 12000],
    'Debt (unit: 10,000)': [0, 0, 10, 50, 200, 30],
    'Approved': ['No', 'Yes', 'Yes', 'No', 'Yes', None]
}
df = pd.DataFrame(data)

X_train = df.drop(columns=['Approved']).iloc[:-1]
y_train = df['Approved'].iloc[:-1]
X_me = df.drop(columns=['Approved']).iloc[-1:]

# --- 2. Prepare the Data ---
# FIX: Feature Scaling (This stops the shapes/size error)
scaler = StandardScaler()
X_array = scaler.fit_transform(X_train.values)
X_me_array = scaler.transform(X_me.values) # Scale 'Me' using the same rules

# FIX: The FutureWarning and String Mapping
# Convert Yes/No to 1/0. Using a lambda or np.where is safest.
y_true = y_train.map({'Yes': 1, 'No': 0}).to_numpy(dtype=float)

# Initialize weights and bias
weights = np.array([0.5, -0.2, 1.0])
b = 0.0
learning_rate = 0.01

def sigmoid(z):
    # Clip prevents overflow (Infinity errors)
    z = np.clip(z, -500, 500) ############
    return 1 / (1 + np.exp(-z))

# --- 3. Training Loop ---
for epoch in range(100):

    # Forward Pass
    Z = X_array @ weights + b
    A = sigmoid(Z)

    # Calculate Error
    error = A - y_true

    # Backward Pass (Gradient Descent)
    # Reshape error to (5, 1) so X_array.T (3, 5) @ error (5, 1) works perfectly
    dw = (X_array.T @ error.reshape(-1, 1)).flatten() / len(X_array)
    db = np.mean(error)

    weights -= learning_rate * dw
    b -= learning_rate * db

# --- 4. Final Prediction for 'Me' ---
Z_me = X_me_array @ weights + b
A_me = sigmoid(Z_me)

print(f"Final Weights: {weights}")
print("-" * 30)

if A_me[0] >= 0.5:
    print(f"Prediction for Me: Loan Approved (Probability: {A_me[0]:.2f})")
else:
    print(f"Prediction for Me: Loan Denied (Probability: {A_me[0]:.2f})")

Final Weights: [ 0.48037386 -0.11574555  0.90595412]
------------------------------
Prediction for Me: Loan Approved (Probability: 0.76)
